In [ ]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import pickle

LOG.propagate = False

ble = get_ble_controller()
ble.connect()

In [ ]:
ble.send_command(CMD.SET_PID_GAINS, "0.07|0.0025|0.04")
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

In [ ]:
kf_msgs = []

def kf_data_handler(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    kf_msgs.append(s)
    print(s)

ble.start_notify(ble.uuid['RX_STRING'], kf_data_handler)
ble.send_command(CMD.START_KF_PID_LOG, "")

In [104]:
ble.stop_notify(ble.uuid['RX_STRING'])

In [ ]:
ble.send_command(CMD.SEND_KF_LOG, "")

In [108]:
ble.stop_notify(ble.uuid['RX_STRING'])

In [ ]:
t_list = []
raw_tof_list = []
est_dist_list = []
est_vel_list = []
u_scaled_list = []

for msg in kf_msgs:
    if msg.startswith("KFPID,"):
        parts = msg.split(",")

        t_list.append(float(parts[1]))
        raw_tof_list.append(float(parts[2]))
        est_dist_list.append(float(parts[3]))
        est_vel_list.append(float(parts[4]))
        u_scaled_list.append(float(parts[5]))

t_ms = np.array(t_list)
t = t_ms / 1000.0

raw_tof_mm = np.array(raw_tof_list)
kf_est_dist_mm = np.array(est_dist_list)
kf_est_vel_mm_s = np.array(est_vel_list)
u_scaled = np.array(u_scaled_list)

print("N =", len(t))
print("t range =", t[0], "to", t[-1], "s")
print("raw_tof range =", raw_tof_mm[0], "to", raw_tof_mm[-1], "mm")
print("kf_est_dist range =", kf_est_dist_mm[0], "to", kf_est_dist_mm[-1], "mm")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

idx = np.arange(len(raw_tof_mm))
target_mm = 304

plt.figure(figsize=(8,5))
plt.step(idx, raw_tof_mm, where='post', linewidth=2, label='ToF')
plt.step(idx, kf_est_dist_mm, where='post', linewidth=2, label='KF')
plt.axhline(y=target_mm, color='red', linestyle='--', linewidth=2, label='Target Distance')

plt.xlabel("count")
plt.ylabel("distance data (mm)")
plt.title("ToF Measurements and KF Output Comparison")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(t, kf_est_vel_mm_s / 1000.0, marker='o', label='KF Estimated Velocity')

plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("KF Estimated Velocity on Robot")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
kf_run_data = {
    "t_ms": t_ms,
    "t_s": t,
    "raw_tof_mm": raw_tof_mm,
    "kf_est_dist_mm": kf_est_dist_mm,
    "kf_est_vel_mm_s": kf_est_vel_mm_s,
    "u_scaled": u_scaled,
    "raw_msgs": kf_msgs
}

with open("lab7_kf_pid_run.pkl", "wb") as f:
    pickle.dump(kf_run_data, f)

print("Saved to lab7_kf_pid_run.pkl")